In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
from scipy.stats import linregress
import mpmath
import matplotlib.pyplot as plt
import time
from numba import njit

# 真实的黎曼零点
mpmath.mp.dps = 15
N_ZEROS = 100
TRUE_ZEROS = np.array([float(mpmath.zetazero(i).imag) for i in range(1, N_ZEROS + 1)])

@njit
def run_universe_exact_splatting(steps, n_bins, u_c, k_opt, c_offset):
    transitions = np.zeros((n_bins, n_bins), dtype=np.float64)
    V = np.zeros(n_bins, dtype=np.float64)
    
    # 物理空间划分：严格均分 [-1.0, 1.0]
    dx = 2.0 / n_bins
    
    # 宇宙起始点 x = 0.5 所在格子
    init_bin = int((0.5 + 1.0) / dx)
    if init_bin >= n_bins: init_bin = n_bins - 1
    V[init_bin] = 1.0 # 初始时刻，概率为 1 的冲激函数
    
    for n in range(1, steps + 1):
        # 冷却退火公式（从右向左降温）
        mu_raw = u_c + k_opt / (np.log(n + c_offset)**2)
        
        # 保险丝
        if mu_raw > 2.0: mu = 2.0
        elif mu_raw < 0.1: mu = 0.1
        else: mu = mu_raw
            
        V_next = np.zeros(n_bins, dtype=np.float64)
        
        # 遍历所有存在概率的格子（上帝视角的波函数演化）
        for i in range(n_bins):
            if V[i] < 1e-12: # 🎯 算力优化：忽略空寂的宇宙区域
                continue
                
            e_left = -1.0 + i * dx
            e_right = e_left + dx
            
            y1 = 1.0 - mu * e_left * e_left
            y2 = 1.0 - mu * e_right * e_right
            
            # 🎯 处理抛物线顶点 (x=0) 包含在当前格子的情况
            if e_left <= 0.0 and e_right >= 0.0:
                y_max = 1.0
                y_min = min(y1, y2)
            else:
                y_max = max(y1, y2)
                y_min = min(y1, y2)
                
            if y_max > 1.0: y_max = 1.0
            if y_min < -1.0: y_min = -1.0
                
            w = y_max - y_min
            
            # 寻找溅射的目标格子范围
            j_min = int((y_min + 1.0) / dx)
            j_max = int((y_max + 1.0) / dx)
            if j_min < 0: j_min = 0
            if j_max >= n_bins: j_max = n_bins - 1
            
            # 🎯 核心逻辑：溅射范围与格子大小的精确映射
            if w <= 1e-12:
                # 极度收缩区，概率全部砸入单一目标格子
                j_center = int(((y_min + y_max)/2.0 + 1.0) / dx)
                if j_center < 0: j_center = 0
                if j_center >= n_bins: j_center = n_bins - 1
                
                flow = V[i]
                V_next[j_center] += flow
                transitions[i, j_center] += flow
            else:
                # 根据重叠面积（Overlap），将概率精确溅射分配
                for j in range(j_min, j_max + 1):
                    E_left = -1.0 + j * dx
                    E_right = E_left + dx
                    
                    overlap_left = max(y_min, E_left)
                    overlap_right = min(y_max, E_right)
                    overlap = overlap_right - overlap_left
                    
                    if overlap > 0.0:
                        prob = overlap / w
                        flow = V[i] * prob
                        V_next[j] += flow
                        transitions[i, j] += flow
                        
        V = V_next # 波函数推进到下一时刻
        
    return transitions

# ================= 🔬 上帝视角探测区 =================
# 我们选取 4 个最具代表性的终点，跑个 10^6 步的快速验证版
test_points_end = [1.35, 1.4011, 1.5437, 1.7548]

TOTAL_STEPS = 10**6 # 注意这里我改成了一百万步，方便你几分钟内看结果
C_OFFSET = 10.0   
DELTA_MU_ABS = 0.02  

results_R2 = []
results_mean_err = []

print(f"🚀 启动【全概率溅射版 (Ulam's Exact Method)】| 冷却: Δμ = -{DELTA_MU_ABS}\n")
print(f"{'目标状态':<15} | {'起始 μ (热)':<12} | {'结束 μ (冷)':<12} | {'R²':<8} | {'平均误差'}")
print("-" * 70)

start_total_t = time.time()

for mu_end in test_points_end:
    start_t = time.time()
    
    # 逆向推导渐近线 U_C 和 K
    t_start_val = 1.0 / (np.log(1 + C_OFFSET)**2)
    t_end_val   = 1.0 / (np.log(TOTAL_STEPS + C_OFFSET)**2)
    k_opt = DELTA_MU_ABS / (t_start_val - t_end_val)
    u_c = mu_end - k_opt * t_end_val
    mu_start = u_c + k_opt * t_start_val
    
    # 注入全概率演化引擎
    trans = run_universe_exact_splatting(TOTAL_STEPS, 5000, u_c, k_opt, C_OFFSET)
    
    P_sparse = sp.csr_matrix(trans, dtype=np.float64)
    row_sums = np.array(P_sparse.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0 
    P_sparse.data /= row_sums[P_sparse.indices]
    
    try:
        eigenvalues, _ = eigs(P_sparse, k=N_ZEROS + 20, which='LM', tol=1e-4)
        phases = np.sort(np.angle(eigenvalues[np.abs(eigenvalues.imag) > 1e-4]))
        unwrapped = np.unwrap(phases)
        
        min_len = min(len(unwrapped), N_ZEROS)
        
        if min_len > 10:
            unwrapped_trunc = unwrapped[:min_len]
            true_zeros_trunc = TRUE_ZEROS[:min_len]
            
            slope, intercept, r_val, _, _ = linregress(unwrapped_trunc, true_zeros_trunc)
            pred = slope * unwrapped_trunc + intercept
            err = np.mean(np.abs(pred - true_zeros_trunc))
            r2 = r_val**2
        else:
            err = 20.0 
            r2 = 0.0
            
    except Exception:
        err = 20.0
        r2 = 0.0
        
    status = ""
    if mu_end == 1.35: status = "(周期死寂)"
    elif mu_end == 1.4011: status = "(费根鲍姆点)"
    elif mu_end == 1.5437: status = "(完美冷寂)"
    elif mu_end == 1.7548: status = "(周期3共振)"
        
    print(f"{mu_end:<7.4f} {status:<8} | {mu_start:<12.4f} | {mu_end:<12.4f} | {r2:<8.4f} | {err:.4f}  ({time.time()-start_t:.1f}s)")

print("-" * 70)
print(f"✅ 精确波函数扫描完成！总耗时: {(time.time()-start_total_t)/60:.2f} 分钟")

🚀 启动【全概率溅射版 (Ulam's Exact Method)】| 冷却: Δμ = -0.02

目标状态            | 起始 μ (热)     | 结束 μ (冷)     | R²       | 平均误差
----------------------------------------------------------------------
1.3500  (周期死寂)   | 1.3700       | 1.3500       | 0.9839   | 5.7958  (4.2s)
1.4011  (费根鲍姆点)  | 1.4211       | 1.4011       | 0.9883   | 5.0255  (5.7s)
1.5437  (完美冷寂)   | 1.5637       | 1.5437       | 0.9885   | 5.2503  (59.3s)
1.7548  (周期3共振)  | 1.7748       | 1.7548       | 0.9851   | 5.8352  (3.5s)
----------------------------------------------------------------------
✅ 精确波函数扫描完成！总耗时: 1.21 分钟
